In [ ]:
from __future__ import annotations

import os
import sys
import copy
import math
import random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

In [ ]:
PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Entrenamiento_SAC import ConfigEntrenamiento, entrenar_sac
from entorno_cartera import EntornoCartera

In [ ]:
def fijar_semillas(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

fijar_semillas(42)

Semilla global fijada en 42


In [ ]:
def encontrar_carpeta_datos() -> Path:
    candidatos = [
        PROJECT_ROOT / "Datos",
        PROJECT_ROOT.parent / "Datos",
        Path("/mnt/data/Datos"),
    ]
    for c in candidatos:
        if c.exists():
            return c
    raise FileNotFoundError("No se encontró la carpeta Datos")

CARPETA_DATOS = encontrar_carpeta_datos()
CARPETA_TRAIN = CARPETA_DATOS / "Train"
CARPETA_VALIDATION = CARPETA_DATOS / "Validation"

print("Datos:", CARPETA_DATOS.resolve())

Carpeta de datos: C:\Users\inigo\Desktop\TFG Info\Datos
Carpeta train: C:\Users\inigo\Desktop\TFG Info\Datos\Train


In [ ]:
def cargar_split(carpeta: Path, nombre_split: str):
    datos_estado = pd.read_csv(
        carpeta / f"datos_estado_{nombre_split}.csv",
        index_col=0,
        parse_dates=True,
    )
    retornos = pd.read_csv(
        carpeta / f"retornos_{nombre_split}.csv",
        index_col=0,
        parse_dates=True,
    )
    rf_diario = pd.read_csv(
        carpeta / f"rf_diario_{nombre_split}.csv",
        index_col=0,
        parse_dates=True,
    )

    if isinstance(rf_diario, pd.DataFrame):
        if rf_diario.shape[1] == 1:
            rf_diario = rf_diario.iloc[:, 0]
        else:
            raise ValueError(f"rf_diario_{nombre_split}.csv debería tener una sola columna")

    rf_diario = rf_diario.astype(float)

    fechas = datos_estado.index.intersection(retornos.index).intersection(rf_diario.index)
    datos_estado = datos_estado.loc[fechas].copy()
    retornos = retornos.loc[fechas].copy()
    rf_diario = rf_diario.loc[fechas].copy()

    return datos_estado, retornos, rf_diario


datos_estado_train, retornos_train, rf_diario_train = cargar_split(CARPETA_TRAIN, "train")
datos_estado_val, retornos_val, rf_diario_val = cargar_split(CARPETA_VALIDATION, "validation")

print("TRAIN:", datos_estado_train.shape, retornos_train.shape, rf_diario_train.shape)
print("VALIDATION:", datos_estado_val.shape, retornos_val.shape, rf_diario_val.shape)

datos_estado_train: (2961, 60)
retornos_train: (2961, 12)
rf_diario_train: (2961,)


In [ ]:
base_config = {
    "semilla": 42,

    "pasos_totales": 10000,
    "tamano_buffer": 50000,
    "tamano_batch": 128,
    "pasos_warmup": 1000,

    "gamma": 0.99,
    "tau": 0.005,

    "lr_actor": 3e-4,
    "lr_criticos": 3e-4,
    "lr_alpha": 3e-4,

    "target_entropy": None,
    "max_concentracion_total_extra": 7.0,

    "frecuencia_actualizacion": 1,
    "actualizaciones_por_step": 1,

    "ventana_log_recompensa": 200,
    "frecuencia_log": 500,

    "reward_scale": 1000.0,
    "offset_target_entropy": -5.0,

    "coste_transaccion": 0.001,
    "valor_inicial": 1000.0,
    "pesos_iniciales": "iguales",
}

base_config

{'semilla': 42,
 'pasos_totales': 10000,
 'tamano_buffer': 50000,
 'tamano_batch': 128,
 'pasos_warmup': 1000,
 'gamma': 0.99,
 'tau': 0.005,
 'lr_actor': 0.0003,
 'lr_criticos': 0.0003,
 'lr_alpha': 0.0003,
 'target_entropy': None,
 'frecuencia_actualizacion': 1,
 'actualizaciones_por_step': 1,
 'ventana_log_recompensa': 200,
 'frecuencia_log': 500,
 'reward_scale': 1000.0,
 'offset_target_entropy': -5.0,
 'max_concentracion_total_extra': 7.0,
 'coste_transaccion': 0.001,
 'valor_inicial': 1000.0,
 'pesos_iniciales': 'iguales'}

In [ ]:
search_spaces = {
    "max_concentracion_total_extra": [5, 7, 10, 15],
    "offset_target_entropy": [-1.0, -3.0, -5.0, -7.0],
    "reward_scale": [500, 1000, 2000],
    "gamma": [0.95, 0.97, 0.99],
    "tau": [0.001, 0.005, 0.01],
    "lr_actor": [1e-4, 3e-4, 5e-4],
    "lr_criticos": [1e-4, 3e-4, 5e-4],
    "lr_alpha": [1e-4, 3e-4, 5e-4],
    "tamano_batch": [64, 128, 256],
}

{'max_concentracion_total_extra': [5, 7, 10, 15],
 'offset_target_entropy': [-1.0, -3.0, -5.0, -7.0],
 'reward_scale': [500, 1000, 2000],
 'gamma': [0.95, 0.97, 0.99],
 'tau': [0.001, 0.005, 0.01],
 'lr_actor': [0.0001, 0.0003, 0.0005],
 'lr_criticos': [0.0001, 0.0003, 0.0005],
 'lr_alpha': [0.0001, 0.0003, 0.0005],
 'tamano_batch': [64, 128, 256]}

In [ ]:
METRICAS_TRAIN = [
    "recompensas",
    "perdida_critic1",
    "perdida_critic2",
    "perdida_actor",
    "alpha",
    "residual_entropia",
    "q_min",
    "q1",
    "q2",
    "target_q",
    "gap_critics",
    "log_prob",
    "log_prob_std",
    "entropia",
    "concentracion_min",
    "concentracion_max",
    "concentracion_media",
    "concentracion_total",
    "concentracion_total_std",
    "accion_min",
    "accion_max",
    "peso_cash",
]

METRICAS_GRAFICOS = [
    "recompensas",
    "perdida_actor",
    "perdida_critic1",
    "perdida_critic2",
    "alpha",
    "q_min",
    "target_q",
    "log_prob",
    "concentracion_total",
    "peso_cash",
]

In [ ]:
def moving_average(x, window=50):
    x = np.asarray(x, dtype=float)
    if window is None or window <= 1:
        return x
    if len(x) == 0:
        return x

    out = np.empty_like(x, dtype=float)
    for i in range(len(x)):
        start = max(0, i - window + 1)
        out[i] = np.nanmean(x[start:i+1])
    return out


def safe_mean(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return np.nan
    return float(np.nanmean(x))


def safe_std(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return np.nan
    return float(np.nanstd(x))

In [ ]:
def construir_entorno(
    datos_estado: pd.DataFrame,
    retornos: pd.DataFrame,
    rf_diario: pd.Series,
    config: Dict[str, Any],
) -> EntornoCartera:
    return EntornoCartera(
        datos_estado=datos_estado,
        retornos_diarios=retornos,
        coste_transaccion=config["coste_transaccion"],
        valor_inicial=config["valor_inicial"],
        pesos_iniciales=config["pesos_iniciales"],
        rf_diario=rf_diario,
    )

In [ ]:
def build_train_config(config_dict: Dict[str, Any]) -> ConfigEntrenamiento:
    return ConfigEntrenamiento(
        semilla=config_dict["semilla"],
        pasos_totales=config_dict["pasos_totales"],
        tamano_buffer=config_dict["tamano_buffer"],
        tamano_batch=config_dict["tamano_batch"],
        pasos_warmup=config_dict["pasos_warmup"],
        gamma=config_dict["gamma"],
        tau=config_dict["tau"],
        lr_actor=config_dict["lr_actor"],
        lr_criticos=config_dict["lr_criticos"],
        lr_alpha=config_dict["lr_alpha"],
        target_entropy=config_dict["target_entropy"],
        max_concentracion_total_extra=config_dict["max_concentracion_total_extra"],
        frecuencia_actualizacion=config_dict["frecuencia_actualizacion"],
        actualizaciones_por_step=config_dict["actualizaciones_por_step"],
        ventana_log_recompensa=config_dict["ventana_log_recompensa"],
        frecuencia_log=config_dict["frecuencia_log"],
        reward_scale=config_dict["reward_scale"],
        offset_target_entropy=config_dict["offset_target_entropy"],
    )

In [ ]:
def politica_determinista_desde_agente(agente, device):
    def funcion_pesos(estado_np):
        estado_t = torch.as_tensor(
            estado_np,
            dtype=torch.float32,
            device=device,
        ).unsqueeze(0)

        with torch.no_grad():
            accion = agente.seleccionar_accion(estado_t, determinista=True)

        return accion.squeeze(0).detach().cpu().numpy()

    return funcion_pesos

In [ ]:
def calcular_metricas_financieras(valor_cartera: pd.Series, rf_diario: pd.Series) -> Dict[str, float]:
    valor_cartera = valor_cartera.astype(float).dropna()
    rend = valor_cartera.pct_change().dropna()

    if len(rend) == 0:
        return {
            "retorno_acumulado": np.nan,
            "retorno_anualizado": np.nan,
            "volatilidad_anualizada": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
        }

    rf = rf_diario.reindex(rend.index).fillna(method="ffill").fillna(0.0)
    exceso = rend - rf

    retorno_acumulado = float(valor_cartera.iloc[-1] / valor_cartera.iloc[0] - 1.0)

    n = len(rend)
    retorno_anualizado = float((valor_cartera.iloc[-1] / valor_cartera.iloc[0]) ** (252.0 / n) - 1.0)

    volatilidad_anualizada = float(rend.std(ddof=1) * np.sqrt(252.0)) if len(rend) > 1 else np.nan

    if exceso.std(ddof=1) > 0:
        sharpe = float((exceso.mean() / exceso.std(ddof=1)) * np.sqrt(252.0))
    else:
        sharpe = np.nan

    rolling_max = valor_cartera.cummax()
    drawdown = valor_cartera / rolling_max - 1.0
    max_drawdown = float(drawdown.min())

    return {
        "retorno_acumulado": retorno_acumulado,
        "retorno_anualizado": retorno_anualizado,
        "volatilidad_anualizada": volatilidad_anualizada,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown,
    }

In [ ]:
def evaluar_en_validation(
    agente,
    config: Dict[str, Any],
    datos_estado_val: pd.DataFrame,
    retornos_val: pd.DataFrame,
    rf_diario_val: pd.Series,
):
    device = next(agente.actor.parameters()).device

    entorno_val = construir_entorno(
        datos_estado=datos_estado_val,
        retornos=retornos_val,
        rf_diario=rf_diario_val,
        config=config,
    )

    policy = politica_determinista_desde_agente(agente, device)
    serie_valor = entorno_val.ejecutar_backtest(policy)
    metricas = calcular_metricas_financieras(serie_valor, rf_diario_val)

    return serie_valor, metricas

In [ ]:
def train_one_run(
    config: Dict[str, Any],
    datos_estado_train: pd.DataFrame,
    retornos_train: pd.DataFrame,
    rf_diario_train: pd.Series,
    datos_estado_val: pd.DataFrame,
    retornos_val: pd.DataFrame,
    rf_diario_val: pd.Series,
):
    fijar_semillas(config["semilla"])

    entorno_train = construir_entorno(
        datos_estado=datos_estado_train,
        retornos=retornos_train,
        rf_diario=rf_diario_train,
        config=config,
    )

    config_train = build_train_config(config)

    history, agente = entrenar_sac(
        entorno=entorno_train,
        config=config_train,
        devolver_agente=True,
    )

    serie_valor_val, metricas_val = evaluar_en_validation(
        agente=agente,
        config=config,
        datos_estado_val=datos_estado_val,
        retornos_val=retornos_val,
        rf_diario_val=rf_diario_val,
    )

    return history, serie_valor_val, metricas_val

In [ ]:
def history_to_long_df(
    history: Dict[str, List[float]],
    experiment_id: str,
    param_name: str,
    param_value: Any,
    run_id: int,
) -> pd.DataFrame:
    max_len = max(len(v) for v in history.values()) if len(history) > 0 else 0

    data = {
        "step": np.arange(max_len),
        "experiment_id": [experiment_id] * max_len,
        "param_name": [param_name] * max_len,
        "param_value": [param_value] * max_len,
        "run_id": [run_id] * max_len,
    }

    for metric in METRICAS_TRAIN:
        vals = list(history.get(metric, []))
        if len(vals) < max_len:
            vals = vals + [np.nan] * (max_len - len(vals))
        data[metric] = vals

    return pd.DataFrame(data)

In [ ]:
def build_summary_row_train(df_run: pd.DataFrame, tail_fraction: float = 0.2) -> Dict[str, Any]:
    out = {}

    if df_run.empty:
        return out

    n = len(df_run)
    tail_n = max(1, int(n * tail_fraction))
    tail_df = df_run.iloc[-tail_n:].copy()

    resumen = [
        "recompensas",
        "perdida_actor",
        "perdida_critic1",
        "perdida_critic2",
        "alpha",
        "q_min",
        "log_prob",
        "concentracion_total",
        "peso_cash",
    ]

    out["n_steps_train"] = n
    out["tail_n_train"] = tail_n

    for m in resumen:
        out[f"{m}_train_final_mean"] = safe_mean(tail_df[m].values)
        out[f"{m}_train_final_std"] = safe_std(tail_df[m].values)

    return out

In [ ]:
def run_experiments(
    param_name: str,
    values: List[Any],
    base_config: Dict[str, Any],
    n_runs: int = 1,
    seed_start: int = 42,
):
    all_logs = []
    resumen_rows = []
    curvas_validation = {}

    for value in values:
        curvas_validation[value] = []

        for run_id in range(n_runs):
            cfg = copy.deepcopy(base_config)
            cfg[param_name] = value
            cfg["semilla"] = seed_start + run_id

            experiment_id = f"{param_name}={value}|run={run_id}"
            print(f"Ejecutando: {experiment_id}")

            history, serie_valor_val, metricas_val = train_one_run(
                config=cfg,
                datos_estado_train=datos_estado_train,
                retornos_train=retornos_train,
                rf_diario_train=rf_diario_train,
                datos_estado_val=datos_estado_val,
                retornos_val=retornos_val,
                rf_diario_val=rf_diario_val,
            )

            df_run = history_to_long_df(
                history=history,
                experiment_id=experiment_id,
                param_name=param_name,
                param_value=value,
                run_id=run_id,
            )
            all_logs.append(df_run)
            curvas_validation[value].append(serie_valor_val)

            row = {
                "experiment_id": experiment_id,
                "param_name": param_name,
                "param_value": value,
                "run_id": run_id,
                "semilla": cfg["semilla"],
            }
            row.update(build_summary_row_train(df_run))
            row.update(metricas_val)

            resumen_rows.append(row)

    logs_df = pd.concat(all_logs, ignore_index=True) if all_logs else pd.DataFrame()
    resumen_df = pd.DataFrame(resumen_rows)

    return logs_df, resumen_df, curvas_validation

In [ ]:
def aggregate_metric(logs_df: pd.DataFrame, metric: str) -> pd.DataFrame:
    if logs_df.empty:
        return pd.DataFrame()

    agg = (
        logs_df.groupby(["param_value", "step"])[metric]
        .agg(["mean", "std", "count"])
        .reset_index()
        .rename(columns={"mean": "metric_mean", "std": "metric_std", "count": "n"})
    )
    return agg

Ejecutando: max_concentration=5|run=0
[Paso     500] recompensa_media(200)=+0.000290 | perdida_critic1=37.709778 | perdida_critic2=37.055992 | perdida_actor=47.358887 | alpha=0.894659 | Q_min=-29.571007 | Q1=-29.450310 | Q2=-29.532145 | target_Q=-29.360674 | gap_critics=0.159552 | log_prob=+20.249258 | log_prob_std=0.807844 | entropia=-20.315893 | conc_min=1.000323 | conc_max=1.680291 | conc_media=1.158512 | conc_total=15.060650 | conc_total_std=0.036792 | accion_min=9.326958e-05 | accion_max=0.443286 | peso_cash=0.071942 | residual_entropia=-4.714859
[Paso    1000] recompensa_media(200)=-0.000951 | perdida_critic1=54.320930 | perdida_critic2=54.412651 | perdida_actor=75.393944 | alpha=0.777875 | Q_min=-60.177567 | Q1=-60.122711 | Q2=-60.117748 | target_Q=-60.745605 | gap_critics=0.114675 | log_prob=+20.395178 | log_prob_std=1.095642 | entropia=-20.684261 | conc_min=1.000005 | conc_max=2.569249 | conc_media=1.200931 | conc_total=15.612108 | conc_total_std=0.040536 | accion_min=2.684597

KeyboardInterrupt: 

In [ ]:
def plot_metric_comparison(
    logs_df: pd.DataFrame,
    metric: str,
    smooth_window: Optional[int] = 50,
    show_std: bool = False,
):
    agg = aggregate_metric(logs_df, metric)
    if agg.empty:
        print(f"No hay datos para {metric}")
        return

    plt.figure(figsize=(10, 5))

    valores = sorted(list(agg["param_value"].dropna().unique()), key=lambda x: str(x))

    for v in valores:
        sub = agg[agg["param_value"] == v].sort_values("step")
        x = sub["step"].values
        y = sub["metric_mean"].values.astype(float)
        y_plot = moving_average(y, smooth_window) if smooth_window and smooth_window > 1 else y

        plt.plot(x, y_plot, label=f"{metric} | {v}")

        if show_std:
            s = sub["metric_std"].fillna(0.0).values.astype(float)
            s_plot = moving_average(s, smooth_window) if smooth_window and smooth_window > 1 else s
            plt.fill_between(x, y_plot - s_plot, y_plot + s_plot, alpha=0.15)

    plt.title(f"Comparativa TRAIN - {metric}")
    plt.xlabel("Paso")
    plt.ylabel(metric)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
def plot_experiment_bundle(logs_df: pd.DataFrame, smooth_window: int = 100):
    for metric in METRICAS_GRAFICOS:
        plot_metric_comparison(
            logs_df=logs_df,
            metric=metric,
            smooth_window=smooth_window,
            show_std=False,
        )

In [ ]:
def plot_validation_curves(curvas_validation: Dict[Any, List[pd.Series]]):
    plt.figure(figsize=(10, 5))

    for valor_hp, lista_series in curvas_validation.items():
        for i, serie in enumerate(lista_series):
            etiqueta = f"{valor_hp}" if i == 0 else None
            plt.plot(serie.index, serie.values, label=etiqueta)

    plt.title("Curvas de valor de cartera en VALIDATION")
    plt.xlabel("Fecha")
    plt.ylabel("Valor cartera")
    plt.legend(title="Valor HP")
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
def save_results(
    logs_df: pd.DataFrame,
    resumen_df: pd.DataFrame,
    output_dir: str = "resultados_experimentos",
    prefix: str = "sac_hpo",
):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    logs_path = out / f"{prefix}_logs.csv"
    resumen_path = out / f"{prefix}_summary.csv"

    logs_df.to_csv(logs_path, index=False)
    resumen_df.to_csv(resumen_path, index=False)

    print("Guardado:")
    print(logs_path.resolve())
    print(resumen_path.resolve())

In [ ]:
# Ejemplo: probar max_concentracion_total_extra
logs_maxc, resumen_maxc, curvas_maxc = run_experiments(
    param_name="max_concentracion_total_extra",
    values=[5, 7, 10, 15],
    base_config=base_config,
    n_runs=1,
    seed_start=42,
)

In [ ]:
resumen_maxc.sort_values(by="sharpe", ascending=False)

In [ ]:
plot_experiment_bundle(logs_maxc, smooth_window=100)

In [ ]:
plot_validation_curves(curvas_maxc)

In [ ]:
save_results(
    logs_df=logs_maxc,
    resumen_df=resumen_maxc,
    output_dir="resultados_experimentos",
    prefix="sac_max_concentracion_total_extra",
)